# Exportar los Parquet oficiales a CSV

**Qué entra:** todos los archivos `.parquet` encontrados recursivamente en `Datos_analiticos_proyecto/`.

**Qué se hace:** convertirlos por bloques para no cargar las tablas grandes completas en memoria. Se conservan las columnas almacenadas y no se agrega un índice artificial.

**Qué sale:** un CSV por Parquet, con el mismo nombre, en `Notebooks/Exportacion CSV/CSV/`. Si hubiera subcarpetas de origen, se conservan para evitar nombres repetidos.

Los Parquet originales no se modifican. Esta es una exportación opcional solicitada para compartir/consultar; **EDA y modelos siguen usando los Parquet oficiales**. Reejecutar reemplaza los CSV correspondientes, sin versiones ni copias adicionales. No hay acceso SQL.

Formato: separador **coma**, codificación **UTF-8 con BOM** (para conservar tildes al importar en Excel), punto decimal y valores ausentes vacíos. CSV no conserva el esquema de tipos; los Parquet siguen siendo la referencia.

**Excel:** una hoja admite 1.048.576 filas. Las tablas mayores se exportan completas, pero no caben en una sola hoja. No se deben recortar para abrirlas; usar una herramienta que admita su tamaño o leerlas por bloques.

## 1. Imports y rutas

La raíz se identifica desde la ubicación del notebook. Cambiar `ORIGEN` solamente si se desea convertir otra carpeta; `DESTINO` es una carpeta separada de los originales.

In [1]:
from pathlib import Path
import csv
import time

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "Datos_analiticos_proyecto").is_dir()
)
ORIGEN = ROOT / "Datos_analiticos_proyecto"
DESTINO = ROOT / "Notebooks" / "Exportacion CSV" / "CSV"
TAMANO_BLOQUE = 100_000
SEPARADOR = ","
CODIFICACION = "utf-8-sig"

archivos = sorted(ORIGEN.rglob("*.parquet"))
if not archivos:
    raise FileNotFoundError(f"No hay archivos Parquet en {ORIGEN}")

DESTINO.mkdir(parents=True, exist_ok=True)
print("Origen:", ORIGEN)
print("Destino:", DESTINO)
print("Archivos encontrados:", len(archivos))

Origen: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Datos_analiticos_proyecto
Destino: C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Notebooks\Exportacion CSV\CSV
Archivos encontrados: 8


## 2. Revisar qué se va a exportar

Se consulta el número de filas y columnas de cada Parquet sin cargar su contenido. No se seleccionan muestras ni se aplican filtros.

In [2]:
inventario = []
for archivo in archivos:
    parquet = pq.ParquetFile(archivo)
    inventario.append({
        "archivo": str(archivo.relative_to(ORIGEN)),
        "filas": parquet.metadata.num_rows,
        "columnas": len(parquet.schema_arrow.names),
    })
display(pd.DataFrame(inventario))

,archivo,filas,columnas
0,base_analitica.parquet,44053,108
1,clima_limpio.parquet,187134,22
2,coeficientes_webflor_limpio.parquet,128567,28
3,curvas_variedad_limpio.parquet,549870,30
4,estimados_semanales_limpio.parquet,176140,31
5,planos_siembra_limpio.parquet,3252152,32
6,produccion_real_limpio.parquet,6389201,24
7,proyecciones_webflor_limpio.parquet,1686034,24


## 3. Exportar las tablas completas

Cada archivo se escribe por bloques en un único CSV, con un solo encabezado. Los tipos Arrow mantienen enteros con nulos sin convertirlos innecesariamente a flotantes. Se comprueba que las filas escritas coincidan con el original. La ejecución sobrescribe sólo los CSV de destino; si se interrumpe, volver a ejecutar esta celda para completar la exportación.

In [3]:
resultados = []

for archivo in archivos:
    inicio = time.perf_counter()
    parquet = pq.ParquetFile(archivo)
    salida = DESTINO / archivo.relative_to(ORIGEN).with_suffix(".csv")
    salida.parent.mkdir(parents=True, exist_ok=True)
    filas_escritas = 0
    primer_bloque = True

    with salida.open("w", encoding=CODIFICACION, newline="") as destino:
        for bloque in parquet.iter_batches(batch_size=TAMANO_BLOQUE):
            tabla = bloque.to_pandas(types_mapper=pd.ArrowDtype, ignore_metadata=True)
            tabla.to_csv(
                destino, index=False, header=primer_bloque,
                sep=SEPARADOR, na_rep="", lineterminator="\n"
            )
            filas_escritas += len(tabla)
            primer_bloque = False

        # Un Parquet vacío también genera su CSV con los nombres de columna.
        if primer_bloque:
            pd.DataFrame(columns=parquet.schema_arrow.names).to_csv(
                destino, index=False, sep=SEPARADOR, lineterminator="\n"
            )

    assert filas_escritas == parquet.metadata.num_rows, f"Faltan filas en {salida.name}"
    resultados.append({
        "archivo": salida.name,
        "filas": filas_escritas,
        "columnas": len(parquet.schema_arrow.names),
        "MB_csv": round(salida.stat().st_size / 1_000_000, 2),
        "segundos": round(time.perf_counter() - inicio, 1),
        "ruta": str(salida.relative_to(ROOT)),
    })
    print(f"OK: {salida.name} | {filas_escritas:,} filas", flush=True)

resumen = pd.DataFrame(resultados)
display(resumen)

OK: base_analitica.csv | 44,053 filas


OK: clima_limpio.csv | 187,134 filas


OK: coeficientes_webflor_limpio.csv | 128,567 filas


OK: curvas_variedad_limpio.csv | 549,870 filas


OK: estimados_semanales_limpio.csv | 176,140 filas


OK: planos_siembra_limpio.csv | 3,252,152 filas


OK: produccion_real_limpio.csv | 6,389,201 filas


OK: proyecciones_webflor_limpio.csv | 1,686,034 filas


,archivo,filas,columnas,MB_csv,segundos,ruta
0,base_analitica.csv,44053,108,38.77,6.1,Notebooks\Exportacion CSV\CSV\base_analitica.csv
1,clima_limpio.csv,187134,22,40.71,4.4,Notebooks\Exportacion CSV\CSV\clima_limpio.csv
2,coeficientes_webflor_limpio.csv,128567,28,24.25,0.7,Notebooks\Exportacion CSV\CSV\coeficientes_web...
3,curvas_variedad_limpio.csv,549870,30,176.61,7.6,Notebooks\Exportacion CSV\CSV\curvas_variedad_...
4,estimados_semanales_limpio.csv,176140,31,67.04,2.2,Notebooks\Exportacion CSV\CSV\estimados_semana...
5,planos_siembra_limpio.csv,3252152,32,889.99,63.0,Notebooks\Exportacion CSV\CSV\planos_siembra_l...
6,produccion_real_limpio.csv,6389201,24,1489.70,217.4,Notebooks\Exportacion CSV\CSV\produccion_real_...
7,proyecciones_webflor_limpio.csv,1686034,24,412.73,69.8,Notebooks\Exportacion CSV\CSV\proyecciones_web...


## 4. Validación final y resumen

Se vuelve a leer cada CSV con un lector que respeta comillas y saltos de línea dentro de campos. Se compara su encabezado y número de registros contra el Parquet. Esta lectura tarda un poco, pero permite comprobar que no se truncaron las tablas. No genera archivos auxiliares.

In [4]:
validaciones = []

for archivo in archivos:
    parquet = pq.ParquetFile(archivo)
    salida = DESTINO / archivo.relative_to(ORIGEN).with_suffix(".csv")
    filas_csv = 0

    with salida.open("r", encoding=CODIFICACION, newline="") as entrada:
        lector = csv.reader(entrada, delimiter=SEPARADOR)
        encabezado = next(lector)
        assert encabezado == parquet.schema_arrow.names, f"Columnas diferentes: {salida.name}"
        for fila in lector:
            assert len(fila) == len(encabezado), f"Registro mal formado: {salida.name}"
            filas_csv += 1

    assert filas_csv == parquet.metadata.num_rows, f"Conteo diferente: {salida.name}"
    validaciones.append({"archivo": salida.name, "filas_verificadas": filas_csv, "estado": "OK"})
    print(f"Verificado: {salida.name}", flush=True)

display(pd.DataFrame(validaciones))
print(f"Listo: {len(archivos)} CSV completos en {DESTINO}")
print("No se modificaron los Parquet ni se consultó SQL.")
print("Los CSV quedan excluidos de Git por la regla *.csv de .gitignore.")

Verificado: base_analitica.csv


Verificado: clima_limpio.csv


Verificado: coeficientes_webflor_limpio.csv


Verificado: curvas_variedad_limpio.csv


Verificado: estimados_semanales_limpio.csv


Verificado: planos_siembra_limpio.csv


Verificado: produccion_real_limpio.csv


Verificado: proyecciones_webflor_limpio.csv


,archivo,filas_verificadas,estado
0,base_analitica.csv,44053,OK
1,clima_limpio.csv,187134,OK
2,coeficientes_webflor_limpio.csv,128567,OK
3,curvas_variedad_limpio.csv,549870,OK
4,estimados_semanales_limpio.csv,176140,OK
5,planos_siembra_limpio.csv,3252152,OK
6,produccion_real_limpio.csv,6389201,OK
7,proyecciones_webflor_limpio.csv,1686034,OK


Listo: 8 CSV completos en C:\Materias y Cosas Técnicas Uniandes\Proyecto de grado\Proyecto de grado 2\Notebooks\Exportacion CSV\CSV
No se modificaron los Parquet ni se consultó SQL.
Los CSV quedan excluidos de Git por la regla *.csv de .gitignore.
